# 24 — The necessity test (E19): leave-EFG-out anchors (T2) and the conditional no-EFG ensemble (T3)

Runs on the curated-block re-solve (`VERSION = "v3"`, after 12 and 18). **T2** (~12 × 1 min): for every design
formulation, the anchor solved with every EFG multiplier at 0 (exactly as the leave-block-out anchors (E17 T3) did
for `efg_out`) → `runs_v3/e19_t2/<formulation_id>/run/portfolio.tif`. Core cells absent from every no-EFG anchor are
EFG-necessary by counterfactual (18c compares them with the adequacy-forced set (E19 T1)).
**T3** (~9 h + a guarded sweep): the full no-EFG ensemble (anchor + 50 MGA members + 50 guarded members per formulation)
— runs ONLY if `spec/v3/e19_gate.json`, written by 18c, says the core is predominantly forced (> 50%). Resumable;
live internet (WLS). Kernel `R (y2y)`.

In [ ]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))
mpath <- pr_refresh_manifest(PROJ, ANALYSIS)
VERSION <- "v3.1"      # v3.1 = the curated block with window-derived targets (study plan v0.17.3)
stopifnot("the necessity test (E19) runs on the curated-block re-solve only (study plan v0.17.2)" = VERSION != "v1")
MANIFEST_REL <- sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION); FREEZE_REL <- sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL <- sprintf("analyses/y2y/runs_%s", VERSION); EFG_SUBDIR_EXPECTED <- paste0("iucn_efg_", sub("\\..*$", "", VERSION))
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot(identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig), nrow(MAN) == 12)
RUNS <- file.path(PROJ, RUNS_REL)
REAL245 <- "input_data/aligned_stack/climate_realizations/macrorefugia_245_2071_2100.tif"
ER <- jsonlite::read_json(file.path(PROJ, "analyses/y2y/spec/e_round_v13.json"))
BLOCKS <- lapply(ER$e17_t3$blocks, unlist); FLOOR_G <- 0.05
ctx585 <- pr_setup(mpath, PROJ); ctx585 <- modifyList(ctx585, pr_ingest(ctx585)); ctx585 <- modifyList(ctx585, pr_planning_units(ctx585))
ctx245 <- pr_setup(mpath, PROJ); ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245)); ctx245 <- modifyList(ctx245, pr_planning_units(ctx245))
efg_names <- ctx585$layers$name[ctx585$layers$role == "feature_efg"]
stopifnot(length(efg_names) > 0, all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), ctx585$layers$path[ctx585$layers$role == "feature_efg"])))
base_for <- function(row) if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585
form_wt  <- function(row) list(w = jsonlite::fromJSON(row$weight_vector), t = jsonlite::fromJSON(row$target_vector))
no_efg <- function(w) { for (f in efg_names) w[[f]] <- 0; w }        # every EFG multiplier -> 0 (E17 T3 convention)
run_single <- function(base_ctx, w, t, out_rel, artifact = "run") {
  done <- file.path(PROJ, out_rel, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s exists -- skipped\n", out_rel)); return(invisible(NULL)) }
  actx <- do.call(pr_override, c(list(base_ctx, targets = t, feature_weight_multipliers = w,
      results_dir = out_rel, results_subdir = artifact, solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)))
  actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing; actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx)); pr_write_outputs(actx); invisible(NULL)
}
cat(sprintf("VERSION %s | %d design formulations | %d EFG features zeroed for the counterfactuals\n", VERSION, nrow(MAN), length(efg_names)))


In [ ]:
# ---- T2: leave-EFG-out anchors, one per design formulation (~1 min each) --------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; wt <- form_wt(row)
  cat(sprintf("== %s (%d/%d)\n", row$formulation_id, i, nrow(MAN)))
  run_single(base_for(row), no_efg(wt$w), wt$t, file.path(RUNS_REL, "e19_t2", row$formulation_id))
}
cat("T2 complete -- next: 18c_e19_analysis (T1 + T2 agreement + the T3 gate)\n")


In [ ]:
# ---- T3 (CONDITIONAL): the no-EFG ensemble -- anchors + MGA + guarded members, EFG multipliers 0 --------------
gate_f <- file.path(PROJ, sprintf("analyses/y2y/spec/%s/e19_gate.json", VERSION))
gate <- if (file.exists(gate_f)) jsonlite::read_json(gate_f) else NULL
if (is.null(gate)) {
  cat("T3 gate not written yet -- run 18c_e19_analysis first (it decides whether the core is predominantly forced)\n")
} else if (!isTRUE(gate$t3_triggered)) {
  cat(sprintf("T3 NOT triggered: forced share of the core %.1f%% (rule: > 50%%) -- the no-EFG ensemble is not run\n", 100 * gate$forced_share_core_all))
} else {
  cat(sprintf("T3 TRIGGERED: forced share of the core %.1f%% -- solving the no-EFG ensemble (~9 h + guarded)\n", 100 * gate$forced_share_core_all))
  for (i in seq_len(nrow(MAN))) {
    row <- MAN[i, ]; wt <- form_wt(row); cd <- file.path(RUNS, "e19_t3", row$formulation_id); dir.create(cd, recursive = TRUE, showWarnings = FALSE)
    cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
    if (file.exists(file.path(cd, "mga_guard_g05.tif"))) { cat("   exists -- skipped\n"); next }
    actx <- pr_override(base_for(row), targets = wt$t, feature_weight_multipliers = no_efg(wt$w),
        results_dir = file.path(RUNS_REL, "e19_t3", row$formulation_id), results_subdir = "mga_build",
        solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
    actx <- modifyList(actx, pr_weights(actx)); actx <- modifyList(actx, pr_targets(actx)); actx <- modifyList(actx, pr_penalty_matrices(actx))
    bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
    cm <- mga_compile(actx); anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
    if (!file.exists(file.path(cd, "mga_g05.tif"))) {
      gen <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested); mga_write(gen, cm, actx$cost, cd, "g05")
    }
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r)); v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
    terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    jsonlite::write_json(list(formulation_id = row$formulation_id, experiment = "E19 T3 no-EFG ensemble", anchor_objective = anchor$z,
                              anchor_gap = anchor$gap, anchor_runtime_s = anchor$runtime, weight_vector = no_efg(wt$w), target_vector = wt$t,
                              k = row$k_requested, g = row$band_gap_g, created_utc = format(Sys.time(), tz = "UTC")),
                         file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
    gg <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested, floors = list(ctx = actx, blocks = BLOCKS, g = FLOOR_G))
    layers <- lapply(seq_len(gg$k), function(j) { rr <- terra::rast(actx$cost); vv <- rep(NA_integer_, terra::ncell(rr)); vv[cm$pu_index] <- as.integer(gg$members[j, ]); terra::values(rr) <- vv; rr })
    s <- terra::rast(layers); names(s) <- sprintf("guard_%02d", seq_len(gg$k))
    terra::writeRaster(s, file.path(cd, "mga_guard_g05.tif"), overwrite = TRUE, datatype = "INT1U", NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    write.csv(gg$certificates, file.path(cd, "certificates_guard.csv"), row.names = FALSE)
    cat(sprintf("   wrote anchor, MGA members, guarded members for %s\n", row$formulation_id))
  }
  cat("T3 complete -- re-run 18c_e19_analysis for the F_noEFG surface\n")
}
